# **Model Fine Tune with PyTorch**

Threre are several ways to do the fine-tune for a model. but we are considering only main two methods in this project.

They are:

* Fine-tune with all the parameters
* Fine-tune with the final layer only

In [35]:
import torch
import torch.nn as nn 
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
from urllib.request import urlopen
import pickle
import tarfile
import io
import os
import tempfile
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


### **Import Dataset**

In [2]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/35t-FeC-2uN1ozOwPs7wFg.gz"

In [3]:
url_open = urlopen(url)
#convert to tar file
tar_file = tarfile.open(fileobj= io.BytesIO(url_open.read()))
temp_file = tempfile.TemporaryDirectory()
tar_file.extractall(temp_file.name)
tar_file.close()

Define IMDB Dataset Class


In [19]:
class IMDBDataset(Dataset):
    
    def __init__(self, root_dir, train = True):
        super().__init__()
        
        self.root_dir = os.path.join(root_dir,"train" if train else "test")
        self.negative_files = [os.path.join(self.root_dir, "neg",f) for f in os.listdir(os.path.join(self.root_dir,"neg")) if f.endswith('.txt')]
        self.positive_files = [os.path.join(self.root_dir,"pos",f) for f in os.listdir(os.path.join(self.root_dir,"pos")) if f.endswith(".txt")]
        
        self.files = self.negative_files + self.positive_files
        self.labels = [0] * len(self.negative_files) + [1] * len(self.positive_files)
        self.pos_idx = len(self.negative_files)
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, index):
        file_path = self.files[index]
        label = self.labels[index]
        
        with open(file_path, "r", encoding='utf-8') as file:
            content = file.read()
        return label, content

In [39]:
root_dir = temp_file.name + '/' + 'imdb_dataset'
train_dataset = IMDBDataset(root_dir,train=True)
test_dataset = IMDBDataset(root_dir,train=False)

In [40]:
start = train_dataset.pos_idx
for item in range(-10,10):
    print(train_dataset[start+item])

(0, "Yeti: Curse of the Snow Demon starts aboard a plane full of American high school teens who are on their way to play a football game in Japan, unfortunately during a fierce thunder storm their plane crashes in the Himalayas. Unlucky really. With some dead & some alive the survivors have to think about themselves & decide to wait it out until help comes. However just when they think their luck couldn't get any worse they soon discover that a huge, hairy Yeti type Abominable Snowman creature wants to kill & eat them all. Trapped, cold, starving & fighting for survival will help reach the stranded teens in time?<br /><br />Yeah, with a title like Yeti: Curse of the Snow Demon it can only mean one thing & that is that someone at the Sci-Fi Channel has made yet another 'Creature Feature' although to give these things a bit of variety the Sci-Fi Channel here in the UK are now dubbing them as a 'Beast Feast'! As if that will make any difference. Directed by Paul Ziller one has to say that

**Label Define**

In [41]:
imdb_label = {0: "negative review", 1: "positive review"}

In [42]:
#define number of classes
num_class = len(set([label for label, text in train_dataset]))
print("Number of classes: ", num_class)

Number of classes:  2


### **Word Embeddings**

In [43]:
class WordEmbedding(nn.Module):
    
    def __init__(self, vocab_size, num_dims, dropout):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocab_size, num_dims)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_tokens):
        word_embed = self.embedding_layer(input_tokens)
        return word_embed

### **Positional Embeddigns**

In [44]:
class PositionalEmbedding(nn.Module):
    
    def __init__(self, max_seq_len, d_model,dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        positions = torch.arange(0,max_seq_len).float().unsqueeze(1)
        
        pe = torch.zeros(max_seq_len,d_model)
        
        div_term = torch.exp(
            torch.arange(0,d_model,2).float() * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
        
    def forward(self, word_embeddings):
        
        word_embed_size = word_embeddings.size(1)
        positional_embeddings = word_embeddings + self.pe[:,0:word_embed_size,:]
        return self.dropout(positional_embeddings)

In [45]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [46]:
tokenizer("hi how are you?").input_ids

[101, 7632, 2129, 2024, 2017, 1029, 102]

In [47]:
#define yield iterator for iteratable tokenization
def yield_tokens(dataset):
    for label,text in dataset:
        yield tokenizer(text)

In [48]:
yield_tokens("hi how are you?")

<generator object yield_tokens at 0x00000293FB499540>

In [49]:
PAD_IDX,UNK_IDX,EOS_IDX  = 0,1,2
special_tokens = ['<pad>','<unk>','<eos>']

In [50]:
vocab_size = tokenizer.vocab_size

#### **Define Data Splits**

In [51]:
num_train = int(0.8 * len(train_dataset))
print(f"num_train:{num_train}")

num_valid = len(train_dataset) - num_train
print(f"num_valid:{num_valid}")

num_train:20000
num_valid:5000


In [52]:
train_split, valid_split = random_split(train_dataset,[num_train,num_valid])

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### **Define Collate Function**

In [70]:
def collate_func (batch):
    
    label_list, text_list = [], []
    
    for label_, text_ in batch:
        label_list.append(label_)
        text_list.append(torch.tensor(tokenizer(text_).input_ids, dtype=torch.int64))
    
    label_list = torch.tensor(label_list, dtype=torch.int64)
    
    #padding the token ids
    text_list = pad_sequence(text_list,batch_first=True, padding_value=PAD_IDX)
    return label_list.to(device), text_list.to(device)

#### **Define Data Loaders**

In [71]:
batch_size = 32

In [72]:
train_dataloader = DataLoader(train_split,batch_size=batch_size,shuffle=True,collate_fn=collate_func)

valid_dataloader = DataLoader(valid_split, batch_size=batch_size, shuffle=True, collate_fn=collate_func)

test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_func)

In [76]:
#test the data loaders
label, text = next(iter(valid_split))
label,text

(0,
 'I was seriously looking forward to seeing this film because it seemed truly promising from the coming attractions: Jim Carrey with Godlike powers was an idea that most definitely worked for me. As a huge fan, I was sure he\'d be supremely in his element with such a promising premise, and what could go wrong? Yesterday, my bubble got burst big-time, boys and girls, because I saw the movie. <br /><br />The first act (where it\'s set up that he hates his life, he\'s a disgruntled employee and a majorly unhappy camper with an ax to grind against God) is serviceable, the second act (where he\'s summoned by God via telephone and receives Powers Almighty) is GREAT - Carrey gets to have fun with his new \'toys\' and it\'s a pleasure to watch, really funny. But the third act is wretched beyond belief.<br /><br />The rot starts setting in after the dinner scene between Bruce and his girlfriend Grace (Jennifer Aniston, who comes off EXCEEDINGLY well in this movie considering her part is mer